In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_H.xpt"
csv_file = DATA_DIR / "ALQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 ALQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ101",
    "ALQ110",
    "ALQ120Q",
    "ALQ120U",
    "ALQ130",
    "ALQ141Q",
    "ALQ141U",
    "ALQ151",
    "ALQ160"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ALQ_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 5924:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 5,924 rows x 10 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# ALQ_H contains NO genuine fractional variables.
#
# Every field is:
#
#   - respondent identifier
#   - questionnaire response code
#   - whole-number quantity
#   - unit code
#
# Therefore all 10 fields can safely become Int64.
#
# IMPORTANT:
#
# This applies ONLY to this file.
#
# Genuine Dietary/Lab/Exam decimal variables in other
# NHANES files must still remain exactly as released.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(DECIMAL_COLUMNS) != 0:

    raise ValueError(
        "ALQ_H should contain no decimal variables."
    )


if len(INTEGER_COLUMNS) != 10:

    raise ValueError(
        "Expected exactly 10 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Never blindly replace this value globally without
# validating where it occurs first.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 ALQ_H fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached ALQ_H.xpt pattern:
#
# ALQ120Q =   882
# ALQ141Q = 2,124
# ALQ160  = 1,454
#
# TOTAL   = 4,460
#
# These are all documented valid zeros.
#

EXPECTED_TINY_COUNTS = {
    "ALQ120Q": 882,
    "ALQ141Q": 2124,
    "ALQ160": 1454
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated ALQ_H attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 4460:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "already existed before correction."
    )


print(
    "\nPASS: Exact 4,460-value tiny-zero "
    "pattern across 3 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT blindly use:
#
#     df.replace(TINY_VALUE, 0)
#
# We restore only the three validated fields.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 4460:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# XPT should contain.
#
# ONLY validated tiny -> zero values are modified.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# This MUST happen before conversion to Int64.
#
# np.round() below is used ONLY for determining whether
# values are whole-number.
#
# It DOES NOT alter data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in ALQ_H. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83729:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83729 validated."
)


# ============================================================
# 12. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# Every ALQ_H variable is proven whole-number.
#
# NO rounding is performed during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 ALQ_H fields converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        "Observed:",
        actual_values
    )


    print(
        "Missing:",
        actual_missing
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. HELPER - RANGE/SPECIAL-CODE CHECK
# ============================================================

def check_range_with_special(
    column,
    range_min,
    range_max,
    expected_range_count,
    expected_missing,
    special_counts=None,
    expected_zero=None
):

    if special_counts is None:
        special_counts = {}


    s = df[column]


    range_count = int(
        s.between(
            range_min,
            range_max
        )
        .sum()
    )


    missing_count = int(
        s.isna()
        .sum()
    )


    zero_count = int(
        s.eq(0)
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )


    print(
        f"Values {range_min}-{range_max}:",
        f"{range_count:,}"
    )


    print(
        "Missing:",
        f"{missing_count:,}"
    )


    print(
        "Zero:",
        f"{zero_count:,}"
    )


    if range_count != expected_range_count:

        raise ValueError(
            f"{column}: expected "
            f"{expected_range_count:,} values "
            f"within {range_min}-{range_max}, "
            f"found {range_count:,}."
        )


    if missing_count != expected_missing:

        raise ValueError(
            f"{column}: expected "
            f"{expected_missing:,} missing, "
            f"found {missing_count:,}."
        )


    if expected_zero is not None:

        if zero_count != expected_zero:

            raise ValueError(
                f"{column}: expected "
                f"{expected_zero:,} zeros, "
                f"found {zero_count:,}."
            )


    for special_code, expected_count in special_counts.items():

        actual_count = int(
            s.eq(special_code)
            .sum()
        )


        print(
            f"{special_code}:",
            actual_count
        )


        if actual_count != expected_count:

            raise ValueError(
                f"{column}: code {special_code} "
                f"expected {expected_count:,}, "
                f"found {actual_count:,}."
            )


    print(
        f"PASS: {column} range/counts validated."
    )


# ============================================================
# 15. CDC CHECK - ALQ101
# ============================================================

#
# Had at least 12 drinks in any one year?
#
# CDC:
#
# 1 = Yes         3,790
# 2 = No          1,623
# 7 = Refused         0
# 9 = Don't know      8
# Missing           503
#

check_frequency(
    "ALQ101",
    {
        1: 3790,
        2: 1623,
        9: 8
    },
    503
)


# ============================================================
# 16. CDC CHECK - ALQ110
# ============================================================

#
# Had at least 12 drinks in lifetime?
#
# CDC:
#
# 1 = Yes           689
# 2 = No            938
# 7 = Refused         0
# 9 = Don't know      4
# Missing         4,293
#

check_frequency(
    "ALQ110",
    {
        1: 689,
        2: 938,
        9: 4
    },
    4293
)


# ============================================================
# 17. CDC CHECK - ALQ120Q
# ============================================================

#
# How often drank alcohol during the past 12 months.
#
# CDC instructions:
#
#     ENTER '0' FOR NEVER.
#
# CDC:
#
# 0-365 = 4,475
# 777 Refused = 0
# 999 Don't know = 4
# Missing = 1,445
#
# Attached XPT:
#
# zero = 882
#

check_range_with_special(
    "ALQ120Q",
    range_min=0,
    range_max=365,
    expected_range_count=4475,
    expected_missing=1445,
    special_counts={
        777: 0,
        999: 4
    },
    expected_zero=882
)


# ============================================================
# 18. CDC CHECK - ALQ120U
# ============================================================

#
# Unit of measure for ALQ120Q.
#
# 1 = Week
# 2 = Month
# 3 = Year
#

check_frequency(
    "ALQ120U",
    {
        1: 1461,
        2: 953,
        3: 1179
    },
    2331
)


# ============================================================
# 19. CDC CHECK - ALQ130
# ============================================================

#
# Average number of alcoholic drinks/day
# on drinking days during past 12 months.
#
# CDC:
#
# 1-25 = 3,593
# 999 Don't know = 3
# Missing = 2,328
#

check_range_with_special(
    "ALQ130",
    range_min=1,
    range_max=25,
    expected_range_count=3593,
    expected_missing=2328,
    special_counts={
        777: 0,
        999: 3
    },
    expected_zero=0
)


# ============================================================
# 20. CDC CHECK - ALQ141Q
# ============================================================

#
# Number of days with 4/5 or more drinks.
#
# CDC instructions:
#
#     ENTER '0' FOR NONE.
#
# CDC:
#
# 0-350 = 3,588
# 999 Don't know = 7
# Missing = 2,329
#
# Attached XPT:
#
# zero = 2,124
#

check_range_with_special(
    "ALQ141Q",
    range_min=0,
    range_max=350,
    expected_range_count=3588,
    expected_missing=2329,
    special_counts={
        777: 0,
        999: 7
    },
    expected_zero=2124
)


# ============================================================
# 21. CDC CHECK - ALQ141U
# ============================================================

#
# Unit of measure for ALQ141Q.
#
# 1 = Week
# 2 = Month
# 3 = Year
#

check_frequency(
    "ALQ141U",
    {
        1: 361,
        2: 264,
        3: 839
    },
    4460
)


# ============================================================
# 22. CDC CHECK - ALQ151
# ============================================================

#
# Ever had 4/5 or more drinks almost every day?
#
# CDC:
#
# 1 = Yes           736
# 2 = No          3,740
# 9 = Don't know      1
# Missing         1,447
#

check_frequency(
    "ALQ151",
    {
        1: 736,
        2: 3740,
        9: 1
    },
    1447
)


# ============================================================
# 23. CDC CHECK - ALQ160
# ============================================================

#
# Number of times during past 30 days respondent
# had 4/5 or more drinks in about two hours.
#
# CDC instructions:
#
#     ENTER '0' FOR NEVER.
#
# CDC:
#
# 0-30 = 1,865
# 999 Don't know = 1
# Missing = 4,058
#
# Attached XPT:
#
# zero = 1,454
#

check_range_with_special(
    "ALQ160",
    range_min=0,
    range_max=30,
    expected_range_count=1865,
    expected_missing=4058,
    special_counts={
        777: 0,
        999: 1
    },
    expected_zero=1454
)


# ============================================================
# 24. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Before repair, attached XPT had:
#
#     ordinary numeric zeros = 0
#
# Therefore ALL numeric zeros after repair must be
# exactly the three validated variables.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated ALQ_H pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 4460:

    raise ValueError(
        "Unexpected final total zero count."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 4,460 validated corrections."
)


# ============================================================
# 25. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY permitted numerical changes are:
#
# ALQ120Q:
#     882 tiny values -> 0
#
# ALQ141Q:
#     2,124 tiny values -> 0
#
# ALQ160:
#     1,454 tiny values -> 0
#
# No other numeric value may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 10 variables match the exact "
    "expected corrected XPT values."
)


# ============================================================
# 26. EXPORT CSV
# ============================================================

#
# Standard fidelity export:
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# ALQ_H happens to contain no genuine decimals, but
# retaining this export convention keeps your NHANES
# workflow safe and consistent.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")


print(
    csv_file
)


# ============================================================
# 27. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 28. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 29. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 30. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 31. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 32. RAW CSV .0 CHECK
# ============================================================

#
# ALQ_H contains no genuine decimal variables.
#
# Therefore it is safe to check ALL 10 fields.
#
# Examples:
#
# SEQN
#     73557.0 -> 73557
#
# ALQ101
#     1.0 -> 1
#
# ALQ120Q
#     0.0 -> 0
#
# ALQ141Q
#     0.0 -> 0
#
# ALQ160
#     0.0 -> 0
#
# DO NOT apply this global rule to mixed decimal files.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any ALQ_H field."
)


# ============================================================
# 33. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly matches "
    "the validated 4,460 corrections."
)


# ============================================================
# 34. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 35. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83729:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "ALQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Alcohol Use"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "ALQ120Q zero restorations:",
    EXPECTED_TINY_COUNTS["ALQ120Q"]
)


print(
    "ALQ141Q zero restorations:",
    EXPECTED_TINY_COUNTS["ALQ141Q"]
)


print(
    "ALQ160 zero restorations:",
    EXPECTED_TINY_COUNTS["ALQ160"]
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 ALQ_H XPT -> CSV
Rows: 5,924
Columns: 10
PASS: Structure = 5,924 rows x 10 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
Whole-number/code fields: 10

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 4,460

Tiny values by variable:
ALQ141Q    2124
ALQ160     1454
ALQ120Q     882

PASS: Exact 4,460-value tiny-zero pattern across 3 variables validated.

--- VALIDATED ZERO RESTORATION ---
ALQ120Q: 882 restored to 0
ALQ141Q: 2,124 restored to 0
ALQ160: 1,454 restored to 0

PASS: 4,460 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
ALQ101: 0 fractional observations
ALQ110: 0 fractional observations
ALQ120Q: 0 fractional observations
ALQ120U: 0 fractional observations
ALQ130: 0 fractional observations
ALQ141Q: 0 fractional observations
ALQ141U: 0 fractional observations
ALQ151: 0 fractional observations
ALQ